In [ ]:
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
text = """
In a quiet meadow surrounded by towering mountains, there was a village where everyone shared stories by the campfire. Each night, the villagers would gather, bringing tales of their adventures, triumphs, and dreams.

One evening, an old traveler arrived, carrying a wooden box. "Inside this box," he said, "is something that will change your lives forever." The villagers eagerly waited as he opened it to reveal... nothing. Confused, they asked, "What is the meaning of this?"

The traveler smiled and said, "The box is empty because the gift is within you. Your imagination, your courage, and your dreams are the treasures you carry. This box is a reminder to open your hearts and embrace the unknown."

From that day forward, the villagers saw the world differently, finding joy in the simplest moments and courage in the face of fear.

---

Far away, in a sprawling desert, a lone scientist named Aria built a machine that could capture the whispers of the wind. "The wind carries stories," she explained to skeptics. "It speaks of the past and the future."

Years passed, and her machine finally worked. It played back the voice of the desert, revealing ancient secrets about how rivers once flowed where there was now only sand. The world marveled at her discovery, realizing the power of listening to what others could not hear.

---

In the distant future, humans lived among the stars. They built cities on planets light-years away, yet they still looked to Earth for inspiration. Captain Zara, a space explorer, led a crew on a mission to find new life. After years of searching, they found a planet with glowing forests and oceans that sang.

The creatures on the planet did not speak but communicated through colors and music. Zara realized that communication was not about words but understanding. The crew stayed, learning the harmony of this world and teaching humanity a new way to connect.

---

Deep in the ocean, a young dolphin named Luna dreamed of exploring the mysterious Abyss. The older dolphins warned her, "The Abyss is dangerous and dark." But Luna was determined.

She swam deeper than anyone had before, discovering glowing corals and creatures that danced like stars. In the darkest depths, she found an ancient pearl that shimmered with the memories of the ocean's past. Luna returned as a hero, proving that the greatest treasures are found when we venture beyond our fears.

The lesson of the ocean: Bravery reveals wonders hidden in the unknown.

---

In a bustling city, a child named Alex loved to watch the trains. "Where do they go?" Alex asked every day. One day, Alex boarded a train and discovered a secret: each station led to a place inside the imagination.

One station was a candy land, another was a jungle filled with friendly tigers, and another was a city of living books that told stories. Alex realized that every journey, whether real or imagined, leads to growth and discovery.

The moral: Every path, no matter how small, has the power to take us somewhere extraordinary.
"""

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 "',-.:?ABCDEFILOSTWYZabcdefghijklmnopqrstuvwxyz
49


In [ ]:
# create a mapping from characters to integers
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]  # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decoder: take a list of integers, output a string

##Example
print(encode("hii there"))
print(decode(encode("hii there")))

[30, 31, 31, 1, 42, 30, 27, 40, 27]
hii there


In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])  # the 1000 characters we looked at earlier will to the GPT look like this

torch.Size([3029]) torch.int64
tensor([ 0, 15, 36,  1, 23,  1, 39, 43, 31, 27, 42,  1, 35, 27, 23, 26, 37, 45,
         1, 41, 43, 40, 40, 37, 43, 36, 26, 27, 26,  1, 24, 47,  1, 42, 37, 45,
        27, 40, 31, 36, 29,  1, 35, 37, 43, 36, 42, 23, 31, 36, 41,  4,  1, 42,
        30, 27, 40, 27,  1, 45, 23, 41,  1, 23,  1, 44, 31, 34, 34, 23, 29, 27,
         1, 45, 30, 27, 40, 27,  1, 27, 44, 27, 40, 47, 37, 36, 27,  1, 41, 30,
        23, 40, 27, 26,  1, 41, 42, 37, 40, 31, 27, 41,  1, 24, 47,  1, 42, 30,
        27,  1, 25, 23, 35, 38, 28, 31, 40, 27,  6,  1, 13, 23, 25, 30,  1, 36,
        31, 29, 30, 42,  4,  1, 42, 30, 27,  1, 44, 31, 34, 34, 23, 29, 27, 40,
        41,  1, 45, 37, 43, 34, 26,  1, 29, 23, 42, 30, 27, 40,  4,  1, 24, 40,
        31, 36, 29, 31, 36, 29,  1, 42, 23, 34, 27, 41,  1, 37, 28,  1, 42, 30,
        27, 31, 40,  1, 23, 26, 44, 27, 36, 42, 43, 40, 27, 41,  4,  1, 42, 40,
        31, 43, 35, 38, 30, 41,  4,  1, 23, 36, 26,  1, 26, 40, 27, 23, 35, 41,
         

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
xb, yb = get_batch('train')
print('inputs:')
print(xb)
print(xb.shape)
print('targets:')
print(yb)
print(yb.shape)

inputs:
tensor([[27,  1, 28, 31, 36, 23, 34, 34],
        [27, 36,  1, 47, 37, 43, 40,  1],
        [27,  1, 41, 42, 23, 40, 41,  6],
        [26,  4,  1, 23, 36, 37, 42, 30]], device='cuda:0')
torch.Size([4, 8])
targets:
tensor([[ 1, 28, 31, 36, 23, 34, 34, 47],
        [36,  1, 47, 37, 43, 40,  1, 30],
        [ 1, 41, 42, 23, 40, 41,  6,  1],
        [ 4,  1, 23, 36, 37, 42, 30, 27]], device='cuda:0')
torch.Size([4, 8])


In [ ]:
for b in range(batch_size):  # batch dimension
    for t in range(block_size):  # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()}, the target is {target}")

when input is [27], the target is 1
when input is [27, 1], the target is 28
when input is [27, 1, 28], the target is 31
when input is [27, 1, 28, 31], the target is 36
when input is [27, 1, 28, 31, 36], the target is 23
when input is [27, 1, 28, 31, 36, 23], the target is 34
when input is [27, 1, 28, 31, 36, 23, 34], the target is 34
when input is [27, 1, 28, 31, 36, 23, 34, 34], the target is 47
when input is [27], the target is 36
when input is [27, 36], the target is 1
when input is [27, 36, 1], the target is 47
when input is [27, 36, 1, 47], the target is 37
when input is [27, 36, 1, 47, 37], the target is 43
when input is [27, 36, 1, 47, 37, 43], the target is 40
when input is [27, 36, 1, 47, 37, 43, 40], the target is 1
when input is [27, 36, 1, 47, 37, 43, 40, 1], the target is 30
when input is [27], the target is 1
when input is [27, 1], the target is 41
when input is [27, 1, 41], the target is 42
when input is [27, 1, 41, 42], the target is 23
when input is [27, 1, 41, 42, 23]

In [ ]:
class TransformerDecoder(nn.Module):
  def __init__(self, config):
    super().__init__()
    assert config.vocab_size is not None
    assert config.block_size is not None
    self.config = config

    self.transformer = nn.ModuleDict(dict(
        wte = nn.Embedding(config.vocab_size, config.n_embd),
        wpe = nn.Embedding(config.block_size, config.n_embd),
        drop = nn.Dropout(config.dropout),
        h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
        ln_f = nn.LayerNorm(config.n_embd),
    ))
    self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)


  def forward(self, idx, targets=None):
    device = idx.device
    b, t = idx.size()
    assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
    pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0)  # shape (1, t)

    # forward the GPT model itself
    tok_emb = self.transformer.wte(idx)  # token embeddings of shape (b, t, n_embd)
    pos_emb = self.transformer.wpe(pos)  # position embeddings of shape (1, t, n_embd)
    x = self.transformer.drop(tok_emb + pos_emb)
    for block in self.transformer.h:
      x = block(x)
    x = self.transformer.ln_f(x)
    logits = self.lm_head(x)  # output logits of shape (b, t, vocab_size)

    # if we are given some desired targets also calculate the loss
    loss = None
    if targets is not None:
      loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

    return logits, loss


In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config) ##Feed Forward

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = F.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        # causal mask to ensure that attention is only applied to the left in the input sequence
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
            .view(1, 1, config.block_size, config.block_size))

        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
      B, T, C = x.size()  # batch size, sequence length, embedding dimensionality (n_embd)
    # calculate query, key, values for all heads in batch and move head forward to be the batch dim
      q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
      k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
      q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)
      v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)  # (B, nh, T, hs)

    # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
      att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
      ##Masking happens here, this wont be there in an encoder
      att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
      att = F.softmax(att, dim=-1)
      att = self.attn_dropout(att)
      y = att @ v  # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
      y = y.transpose(1, 2).contiguous().view(B, T, C)  # re-assemble all head outputs side by side

    # output projection
      y = self.resid_dropout(self.c_proj(y))
      return y

In [ ]:
class Config:
    vocab_size = len(chars)  # Number of unique characters
    block_size = 16  # Context window (length of sequences)
    n_embd = 64  # Embedding size (reduced for a small dataset)
    n_layer = 3  # Number of Transformer layers
    n_head = 4  # Number of attention heads
    dropout = 0.1  # Dropout for regularization

In [ ]:
from torch.optim import AdamW

config = Config()
model = TransformerDecoder(config).to(device)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

batch_size = 4
block_size = config.block_size
epochs = 100

In [ ]:
for epoch in range(epochs):
    model.train()
    train_loss = 0

    # Train for 50 batches per epoch
    for _ in range(50):
        xb, yb = get_batch('train')  # Fetch batch (batch_size=4, block_size=16)
        logits, loss = model(xb, yb)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Average loss over the epoch
    avg_train_loss = train_loss / 50
    print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {avg_train_loss:.4f}")

Epoch 1/100, Train Loss: 1.6216
Epoch 2/100, Train Loss: 1.6648
Epoch 3/100, Train Loss: 1.6357
Epoch 4/100, Train Loss: 1.5983
Epoch 5/100, Train Loss: 1.6469
Epoch 6/100, Train Loss: 1.6379
Epoch 7/100, Train Loss: 1.6264
Epoch 8/100, Train Loss: 1.6188
Epoch 9/100, Train Loss: 1.6113
Epoch 10/100, Train Loss: 1.6197
Epoch 11/100, Train Loss: 1.6154
Epoch 12/100, Train Loss: 1.6057
Epoch 13/100, Train Loss: 1.6102
Epoch 14/100, Train Loss: 1.5784
Epoch 15/100, Train Loss: 1.5953
Epoch 16/100, Train Loss: 1.5881
Epoch 17/100, Train Loss: 1.5601
Epoch 18/100, Train Loss: 1.5772
Epoch 19/100, Train Loss: 1.5568
Epoch 20/100, Train Loss: 1.5683
Epoch 21/100, Train Loss: 1.6000
Epoch 22/100, Train Loss: 1.5286
Epoch 23/100, Train Loss: 1.5413
Epoch 24/100, Train Loss: 1.5502
Epoch 25/100, Train Loss: 1.5227
Epoch 26/100, Train Loss: 1.5034
Epoch 27/100, Train Loss: 1.5137
Epoch 28/100, Train Loss: 1.5562
Epoch 29/100, Train Loss: 1.5136
Epoch 30/100, Train Loss: 1.5075
Epoch 31/100, Train

In [ ]:
def generate_text(model, start_text, max_new_tokens, stoi, itos, device='cuda'):
    # Encode the starting text
    input_ids = torch.tensor([stoi[ch] for ch in start_text], dtype=torch.long).unsqueeze(0).to(device)

    model.eval()  # Put the model in evaluation mode
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Ensure the sequence length does not exceed block size
            if input_ids.size(1) > model.config.block_size:
                input_ids = input_ids[:, -model.config.block_size:]

            # Forward pass through the model
            logits, _ = model(input_ids)

            # Get the logits for the last token in the sequence
            logits = logits[:, -1, :]  # Shape: (1, vocab_size)

            # Convert logits to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token from the probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # Shape: (1, 1)

            # Append the sampled token to the input sequence
            input_ids = torch.cat((input_ids, next_token), dim=1)

    # Decode the generated sequence back to text
    output_text = ''.join([itos[idx] for idx in input_ids[0].tolist()])
    return output_text

In [ ]:
start_text = "To be"
max_new_tokens = 50  # Generate 50 new characters

# Generate text
generated_text = generate_text(model, start_text, max_new_tokens, stoi, itos, device)
print(generated_text)

a, her but decrea
